phase 4: model selection

training and comparing random forest and xgboost baselines.

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# set style
sns.set_theme(style="whitegrid")

In [ ]:
# load processed data
print('--- loading processed data ---')
X_train = pd.read_csv('../data/processed/X_train.csv')
X_val = pd.read_csv('../data/processed/X_val.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_val = pd.read_csv('../data/processed/y_val.csv').squeeze()

print(f'train shape: {X_train.shape}')
print(f'val shape: {X_val.shape}')

In [ ]:
# train random forest
print('--- random forest baseline ---')
rf = RandomForestClassifier(random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# predict and evaluate
rf_preds = rf.predict(X_val)
rf_probs = rf.predict_proba(X_val)[:, 1]

rf_acc = accuracy_score(y_val, rf_preds)
rf_f1 = f1_score(y_val, rf_preds)
rf_auc = roc_auc_score(y_val, rf_probs)

print(f'accuracy: {rf_acc:.4f}')
print(f'f1 score: {rf_f1:.4f}')
print(f'roc-auc:  {rf_auc:.4f}\n')
print(classification_report(y_val, rf_preds))

In [ ]:
# train xgboost
print('--- xgboost baseline ---')
xgb = XGBClassifier(random_state=42, n_jobs=-1, eval_metric='logloss')
xgb.fit(X_train, y_train)

# predict and evaluate
xgb_preds = xgb.predict(X_val)
xgb_probs = xgb.predict_proba(X_val)[:, 1]

xgb_acc = accuracy_score(y_val, xgb_preds)
xgb_f1 = f1_score(y_val, xgb_preds)
xgb_auc = roc_auc_score(y_val, xgb_probs)

print(f'accuracy: {xgb_acc:.4f}')
print(f'f1 score: {xgb_f1:.4f}')
print(f'roc-auc:  {xgb_auc:.4f}\n')
print(classification_report(y_val, xgb_preds))

In [ ]:
# compare models visually
print('--- model comparison ---')
metrics = pd.DataFrame({
    'model': ['random forest', 'xgboost'],
    'f1_score': [rf_f1, xgb_f1],
    'roc_auc': [rf_auc, xgb_auc]
})

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=metrics, x='model', y='f1_score', ax=axes[0])
axes[0].set_title('f1 score comparison')
axes[0].set_ylim(0, 1)

sns.barplot(data=metrics, x='model', y='roc_auc', ax=axes[1])
axes[1].set_title('roc-auc comparison')
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()